# Проверка фильтра на чистых примерах

Блокнот загружает актуальный код из ветки `codex/context-authority-filter-v2` в новую папку и показывает номер коммита. Проверка охватывает обычные документы, пары запросов и сохранение текста при фильтрации.

Запусти по порядку ячейки **«Загрузить код ветки»** и **«Проверить фильтр»**. Для этого достаточно стандартной среды Colab с процессором. Результат показывает решения самого фильтра. Для просмотра сохранённого эксперимента на языковой модели есть отдельный блокнот сравнения.

In [ ]:
# @title Загрузить код ветки
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

REPOSITORY = "https://github.com/bissonette/analysis-of-injections-using-retrieved-context.git"
BRANCH = "codex/context-authority-filter-v2"

project = None
checkout = Path(mkdtemp(prefix="ctxauth-clean-")) / "project"
subprocess.run(
    ["git", "clone", "--quiet", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(checkout)],
    check=True,
)
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=checkout, text=True).strip()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-r", str(checkout / "requirements.txt")],
    check=True,
)
project = checkout
print(f"Ветка: {BRANCH}")
print(f"Коммит: {revision}")

In [ ]:
# @title Проверить фильтр
if globals().get("project") is None:
    raise RuntimeError("Сначала загрузи код из ветки.")

subprocess.run(
    [sys.executable, "-m", "unittest", "tests.test_filter_context", "-v"],
    cwd=project,
    check=True,
)

check_clean_corpus = """
from collections import Counter

from ctxauth.authority import AuthorityAnalyzer, ContextAuthorityController
from ctxauth.data.loaders import load_builtin_clean

controller = ContextAuthorityController(AuthorityAnalyzer(mode="rules"))
samples = load_builtin_clean()
if not samples or any(s.is_injected for s in samples):
    raise ValueError("Для этой проверки нужна встроенная чистая выборка.")
counts = Counter(controller.filter(s.query, [s.as_chunk()], intent=s.intent).decisions[0].action.value
                 for s in samples)
print(f"Чистых примеров: {len(samples)}")
print(f"Пропущено целиком: {counts['pass']}")
print(f"Очищено: {counts['sanitize']}")
print(f"Исключено: {counts['block']}")
print(f"Затронуто фильтром: {counts['sanitize'] + counts['block']} из {len(samples)}")
"""
subprocess.run([sys.executable, "-c", check_clean_corpus], cwd=project, check=True)